In [1]:
# ==========================================
# CELL 1: AUDIT & PEMBERSIHAN DATASET LOKASI
# ==========================================
import pandas as pd
import os
import re

# 1. SETUP PATH FILE
path_lama = "../../data/dataset_tiket_lengkap_lokasi.csv"
path_baru = "../../data/dataset_tiket_lengkap_lokasi_revisi.csv"

print(f"📥 Membaca data lama dari: {path_lama}...")

try:
    # Load dataset
    df = pd.read_csv(path_lama, sep='|', on_bad_lines='skip')
    total_awal = len(df)
    print(f"📊 Total baris awal: {total_awal} baris\n")

    # 2. PEMBERSIHAN DASAR (DROP NAN & DUPLIKAT IDENTIK)
    kolom_wajib = ['teks_keluhan_awam', 'teks_laporan_teknisi', 'lokasi_gedung', 'lokasi_lantai', 'lokasi_zona', 'severity']
    df = df.dropna(subset=kolom_wajib)
    df = df.drop_duplicates(subset=['teks_keluhan_awam'])
    
    # Pastikan severity hanya 4 kelas (buang jika AI salah geser kolom)
    df = df[df['severity'].isin(['Ringan', 'Sedang', 'Berat', 'Fatal'])]
    
    print(f"🧹 Setelah drop NaN, Duplikat, dan Error Kolom: {len(df)} baris")

    # 3. FILTER "HARGA MATI": WAJIB ADA KATA LOKASI DI TEKS KELUHAN
    # Kita gunakan RegEx sederhana untuk mendeteksi apakah AI menyebutkan gedung, lantai, dan zona
    print("🔍 Melakukan pemindaian konteks lokasi pada teks keluhan...")
    
    mask_lantai = df['teks_keluhan_awam'].str.contains(r'lantai|lt\.', flags=re.IGNORECASE, na=False)
    mask_gedung = df['teks_keluhan_awam'].str.contains(r'gedung|gd\.', flags=re.IGNORECASE, na=False)
    mask_zona   = df['teks_keluhan_awam'].str.contains(r'zona', flags=re.IGNORECASE, na=False)
    
    # Hanya simpan baris yang teks keluhannya punya KETIGA kata kunci tersebut
    df_bersih = df[mask_lantai & mask_gedung & mask_zona].copy()
    
    total_bersih = len(df_bersih)
    baris_dibuang = len(df) - total_bersih
    
    print(f"🗑️ Ditemukan {baris_dibuang} baris yang AI-nya 'malas' sebut lokasi lengkap. (DIBUANG!)")
    print(f"💎 Sisa data 'Grade A' (Lokasi Lengkap & Bersih): {total_bersih} baris\n")

    # 4. SIMPAN KE FILE REVISI
    df_bersih.to_csv(path_baru, sep='|', index=False)
    print(f"💾 BERHASIL! Data revisi disimpan ke: {path_baru}")
    
    # Tampilkan cuplikan data yang selamat
    print("-" * 50)
    print("✨ Cuplikan Teks Keluhan yang Selamat (Pasti ada lokasinya):")
    for teks in df_bersih['teks_keluhan_awam'].head(3):
        print(f"-> \"{teks}\"")

except FileNotFoundError:
    print(f"❌ Error: File {path_lama} tidak ditemukan. Pastikan path-nya benar!")
except Exception as e:
    print(f"❌ Terjadi kesalahan: {e}")

📥 Membaca data lama dari: ../../data/dataset_tiket_lengkap_lokasi.csv...
📊 Total baris awal: 798 baris

🧹 Setelah drop NaN, Duplikat, dan Error Kolom: 389 baris
🔍 Melakukan pemindaian konteks lokasi pada teks keluhan...
🗑️ Ditemukan 226 baris yang AI-nya 'malas' sebut lokasi lengkap. (DIBUANG!)
💎 Sisa data 'Grade A' (Lokasi Lengkap & Bersih): 163 baris

💾 BERHASIL! Data revisi disimpan ke: ../../data/dataset_tiket_lengkap_lokasi_revisi.csv
--------------------------------------------------
✨ Cuplikan Teks Keluhan yang Selamat (Pasti ada lokasinya):
-> "Kamera CCTV di Gedung Utama lantai 15 zona timur tiba-tiba netes dan agak berisik karena overheat, sepertinya ada human error yang menyebabkan kerusakan ini"
-> "Kamera CCTV di Gedung Utama lantai 15 zona timur mengalami kerusakan karena overheat yang disebabkan oleh human error, perlu perbaikan segera"
-> "DVR CCTV di Gedung B Lantai 10 Zona Timur netes karena getaran berlebihan dan tegangan tidak stabil"
